In [0]:
df=spark.read.format("csv").\
            options(header='true', inferSchema='true').\
            load('/Volumes/databricksdan/bronze/bronze_volume/customers/dim_customers.csv')

In [0]:
df.display(5)

In [0]:
df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df=df.withColumn("name",upper(col("name")))

In [0]:
df.show(5)

In [0]:
df=df.withColumn("domain",split(col("email"),"@")[1])

In [0]:
df.show(5)

In [0]:
display(df.groupBy("domain").agg(count(col("customer_id")).alias("total_customer")).sort(col("total_customer").desc()))

Databricks visualization. Run in Databricks to view.

In [0]:
df=df.withColumn("processDate",current_timestamp())
display(df)


In [0]:
from delta.tables import DeltaTable

In [0]:
if spark.catalog.tableExists("databricksdan.silver.customer_enr"):
    dlt_obj=DeltaTable.forName(spark,"databricksdan.silver.customer_enr")
    dlt_obj.alias('tgt').merge(df.alias('src'),'tgt.customer_id=src.customer_id').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df.write.format("delta").mode("append").saveAsTable("databricksdan.silver.customer_enr")